# Self-Supervised Learning: Learning Without Labels

## 1. Introduction

**Self-supervised learning** is a powerful paradigm that allows models to learn useful representations from unlabeled data by creating **pretext tasks** — artificial tasks where labels are automatically generated from the data itself.

### Why Self-Supervised Learning Matters

Labeled data is expensive and time-consuming to collect. Self-supervised learning solves this by:
- **Creating free supervision** from the data structure itself
- **Learning transferable representations** that work well on downstream tasks
- **Scaling to massive datasets** without manual labeling

### What We'll Learn

We'll explore five major self-supervised learning approaches:

1. **Rotation Prediction** — Predict how an image has been rotated
2. **Jigsaw Puzzles** — Predict the correct ordering of shuffled image patches
3. **Masked Autoencoding** — Reconstruct masked parts of images
4. **SimCLR** — Learn representations through contrastive learning
5. **BERT-Style Masking** — Predict masked words in text sequences

Each approach teaches the model to understand the structure of data without explicit labels.

## 2. Setup

We'll import PyTorch, our shared library utilities, and visualization tools.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms

import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List
import random

from aiml_notebooks import (
    get_device, 
    set_seed,
    create_dataset,
    MNIST_MEAN,
    MNIST_STD,
    imshow_normalized,
    plot_image_grid
)

%load_ext autoreload
%autoreload 2

Set random seed for reproducibility and configure device.

In [ ]:
set_seed(42)
device = get_device()
print(f"Using device: {device}")

## 3. Rotation Prediction

### Intuition

**Rotation prediction** is a pretext task where we rotate images by 0°, 90°, 180°, or 270°, and train a model to predict the rotation angle. This forces the model to learn spatial features and object orientation.

Why does this work?
- The model must understand object shapes and spatial relationships to determine rotation
- These learned features transfer well to other vision tasks
- Labels are free — we know exactly how much we rotated each image!

### Load MNIST Dataset

We'll use MNIST digits for our rotation prediction task.

In [ ]:
# Load MNIST dataset
full_dataset, train_dataset, _ = create_dataset("mnist", splits=[0.9, 0.1])
print(f"Training samples: {len(train_dataset)}")

### Create Rotation Dataset

We'll create a dataset wrapper that randomly rotates images and provides the rotation angle as the label.

In [ ]:
class RotationDataset(Dataset):
    """Dataset that applies random rotations (0, 90, 180, 270 degrees) to images."""
    
    def __init__(self, base_dataset):
        self.base_dataset = base_dataset
        self.rotations = [0, 90, 180, 270]  # Possible rotation angles
    
    def __len__(self):
        return len(self.base_dataset)
    
    def __getitem__(self, idx):
        image, _ = self.base_dataset[idx]  # Ignore original label
        
        # Randomly select a rotation
        rotation_idx = random.randint(0, 3)
        angle = self.rotations[rotation_idx]
        
        # Rotate the image (PyTorch expects angle in degrees)
        rotated = transforms.functional.rotate(image, angle)
        
        return rotated, rotation_idx  # Return rotation class (0-3)

rotation_train = RotationDataset(train_dataset)
rotation_loader = DataLoader(rotation_train, batch_size=128, shuffle=True)

### Visualize Rotated Images

Let's see what our rotation pretext task looks like.

In [ ]:
# Get a sample batch
images, labels = next(iter(rotation_loader))

# Display first 8 images with their rotation labels
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
rotation_names = ['0°', '90°', '180°', '270°']

for i, ax in enumerate(axes.flat):
    img = images[i].squeeze().numpy()
    ax.imshow(img, cmap='gray')
    ax.set_title(f'Rotation: {rotation_names[labels[i]]}')
    ax.axis('off')

plt.tight_layout()
plt.show()

print("The model must predict which rotation was applied to each image.")

### Build Rotation Predictor

A simple CNN that classifies images into 4 rotation classes.

In [ ]:
class RotationPredictor(nn.Module):
    """Simple CNN for predicting image rotation."""
    
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 3 * 3, 128)
        self.fc2 = nn.Linear(128, 4)  # 4 rotation classes
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # 28x28 -> 14x14
        x = self.pool(F.relu(self.conv2(x)))  # 14x14 -> 7x7
        x = self.pool(F.relu(self.conv3(x)))  # 7x7 -> 3x3
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

model = RotationPredictor().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

### Train Rotation Predictor

Train the model to predict rotations using standard supervised learning.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train for a few epochs
epochs = 3
losses = []

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    correct = 0
    total = 0
    
    for images, labels in rotation_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    avg_loss = epoch_loss / len(rotation_loader)
    accuracy = 100.0 * correct / total
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

### Key Insight: Rotation Prediction

The model learns to recognize spatial features and object parts to determine rotation. These learned features can be transferred to other tasks like classification or detection — without ever seeing real labels!

## 4. Jigsaw Puzzles

### Intuition

**Jigsaw puzzles** split images into patches, shuffle them, and train a model to predict the correct ordering. This forces the model to learn spatial context and object parts.

Key insight: To solve jigsaw puzzles, the model must understand:
- What objects look like
- How parts relate to each other spatially
- Continuity at patch boundaries

### Create Jigsaw Dataset

We'll split images into 4 patches (2x2 grid) and create a permutation prediction task.

In [ ]:
class JigsawDataset(Dataset):
    """Dataset that splits images into patches and shuffles them."""
    
    def __init__(self, base_dataset, n_patches=4):
        self.base_dataset = base_dataset
        self.n_patches = n_patches
        
        # Define a fixed set of permutations to predict
        # For 4 patches, we'll use a subset of possible permutations
        self.permutations = [
            [0, 1, 2, 3],  # Original order
            [1, 0, 3, 2],  # Swap columns
            [2, 3, 0, 1],  # Swap rows
            [3, 2, 1, 0],  # Reverse
            [0, 2, 1, 3],  # Diagonal swap 1
            [1, 3, 0, 2],  # Diagonal swap 2
        ]
    
    def __len__(self):
        return len(self.base_dataset)
    
    def __getitem__(self, idx):
        image, _ = self.base_dataset[idx]
        
        # Split into 2x2 patches
        c, h, w = image.shape
        patch_h, patch_w = h // 2, w // 2
        
        patches = []
        for i in range(2):
            for j in range(2):
                patch = image[:, i*patch_h:(i+1)*patch_h, j*patch_w:(j+1)*patch_w]
                patches.append(patch)
        
        # Select random permutation
        perm_idx = random.randint(0, len(self.permutations) - 1)
        permutation = self.permutations[perm_idx]
        
        # Apply permutation
        shuffled_patches = [patches[i] for i in permutation]
        
        # Stack patches along channel dimension
        shuffled = torch.cat(shuffled_patches, dim=0)  # Shape: (4, 14, 14)
        
        return shuffled, perm_idx

jigsaw_train = JigsawDataset(train_dataset)
jigsaw_loader = DataLoader(jigsaw_train, batch_size=128, shuffle=True)

### Visualize Jigsaw Puzzles

Let's see how images are split and shuffled.

In [ ]:
# Get original image and jigsaw version
original_img, _ = train_dataset[0]
jigsaw_img, perm_idx = jigsaw_train[0]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Show original
axes[0].imshow(original_img.squeeze(), cmap='gray')
axes[0].set_title('Original Image')
axes[0].axis('off')

# Show jigsaw patches in 2x2 grid
jigsaw_grid = np.zeros((28, 28))
for i in range(2):
    for j in range(2):
        patch_idx = i * 2 + j
        patch = jigsaw_img[patch_idx].numpy()
        jigsaw_grid[i*14:(i+1)*14, j*14:(j+1)*14] = patch

axes[1].imshow(jigsaw_grid, cmap='gray')
axes[1].set_title(f'Shuffled Patches (Permutation {perm_idx})')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print("The model must predict which permutation was applied.")

### Build Jigsaw Solver

A CNN that takes 4 patches as input and predicts which permutation was applied.

In [ ]:
class JigsawSolver(nn.Module):
    """CNN for predicting jigsaw puzzle permutation."""
    
    def __init__(self, n_permutations=6):
        super().__init__()
        self.conv1 = nn.Conv2d(4, 32, 3, padding=1)  # 4 patches as input channels
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 3 * 3, 128)
        self.fc2 = nn.Linear(128, n_permutations)
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # 14x14 -> 7x7
        x = self.pool(F.relu(self.conv2(x)))  # 7x7 -> 3x3
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

jigsaw_model = JigsawSolver().to(device)
print(f"Parameters: {sum(p.numel() for p in jigsaw_model.parameters()):,}")

### Train Jigsaw Solver

Train the model to recognize permutations.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(jigsaw_model.parameters(), lr=0.001)

epochs = 3

for epoch in range(epochs):
    jigsaw_model.train()
    epoch_loss = 0
    correct = 0
    total = 0
    
    for images, labels in jigsaw_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = jigsaw_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    avg_loss = epoch_loss / len(jigsaw_loader)
    accuracy = 100.0 * correct / total
    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

### Key Insight: Jigsaw Puzzles

By solving jigsaw puzzles, the model learns to recognize object parts and their spatial relationships. This teaches visual semantics — what goes with what — without any labels!

## 5. Masked Autoencoding

### Intuition

**Masked autoencoding** randomly masks portions of an image and trains a model to reconstruct the missing parts. This is similar to what humans do when we "fill in the blanks" — we use context to infer what's missing.

Recent breakthrough: **MAE (Masked Autoencoders)** by Meta AI showed that masking 75% of image patches and reconstructing them produces excellent visual representations.

### Create Masked Autoencoding Dataset

We'll randomly mask patches of images and train a model to reconstruct them.

In [ ]:
class MaskedAutoencodingDataset(Dataset):
    """Dataset that randomly masks image patches."""
    
    def __init__(self, base_dataset, mask_ratio=0.5):
        self.base_dataset = base_dataset
        self.mask_ratio = mask_ratio
    
    def __len__(self):
        return len(self.base_dataset)
    
    def __getitem__(self, idx):
        image, _ = self.base_dataset[idx]
        
        # Create a mask (1 = keep, 0 = mask)
        c, h, w = image.shape
        mask = torch.rand(h, w) > self.mask_ratio
        mask = mask.float().unsqueeze(0)  # Add channel dimension
        
        # Apply mask (set masked pixels to 0)
        masked_image = image * mask
        
        return masked_image, image  # Return masked input and original target

mae_train = MaskedAutoencodingDataset(train_dataset, mask_ratio=0.5)
mae_loader = DataLoader(mae_train, batch_size=128, shuffle=True)

### Visualize Masked Images

Let's see what masked images look like.

In [ ]:
# Get samples
masked_imgs, original_imgs = next(iter(mae_loader))

fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for i in range(4):
    # Original
    axes[0, i].imshow(original_imgs[i].squeeze(), cmap='gray')
    axes[0, i].set_title('Original')
    axes[0, i].axis('off')
    
    # Masked
    axes[1, i].imshow(masked_imgs[i].squeeze(), cmap='gray')
    axes[1, i].set_title('Masked (50%)')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

print("The model must reconstruct the original image from the masked version.")

### Build Masked Autoencoder

A simple CNN autoencoder that reconstructs masked images.

In [ ]:
class MaskedAutoencoder(nn.Module):
    """Simple CNN autoencoder for reconstructing masked images."""
    
    def __init__(self):
        super().__init__()
        # Encoder
        self.enc1 = nn.Conv2d(1, 32, 3, padding=1)
        self.enc2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        
        # Decoder
        self.dec1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec2 = nn.ConvTranspose2d(32, 16, 2, stride=2)
        self.dec3 = nn.Conv2d(16, 1, 3, padding=1)
    
    def forward(self, x):
        # Encode
        x = self.pool(F.relu(self.enc1(x)))  # 28 -> 14
        x = self.pool(F.relu(self.enc2(x)))  # 14 -> 7
        
        # Decode
        x = F.relu(self.dec1(x))  # 7 -> 14
        x = F.relu(self.dec2(x))  # 14 -> 28
        x = self.dec3(x)
        return x

mae_model = MaskedAutoencoder().to(device)
print(f"Parameters: {sum(p.numel() for p in mae_model.parameters()):,}")

### Train Masked Autoencoder

Train the model to reconstruct original images from masked inputs.

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(mae_model.parameters(), lr=0.001)

epochs = 3

for epoch in range(epochs):
    mae_model.train()
    epoch_loss = 0
    
    for masked_imgs, original_imgs in mae_loader:
        masked_imgs = masked_imgs.to(device)
        original_imgs = original_imgs.to(device)
        
        optimizer.zero_grad()
        reconstructed = mae_model(masked_imgs)
        loss = criterion(reconstructed, original_imgs)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(mae_loader)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")

### Visualize Reconstructions

See how well the model fills in masked regions.

In [ ]:
mae_model.eval()
with torch.no_grad():
    masked_imgs, original_imgs = next(iter(mae_loader))
    masked_imgs = masked_imgs.to(device)
    reconstructed = mae_model(masked_imgs).cpu()

fig, axes = plt.subplots(3, 4, figsize=(12, 9))

for i in range(4):
    # Original
    axes[0, i].imshow(original_imgs[i].squeeze(), cmap='gray')
    axes[0, i].set_title('Original')
    axes[0, i].axis('off')
    
    # Masked input
    axes[1, i].imshow(masked_imgs[i].cpu().squeeze(), cmap='gray')
    axes[1, i].set_title('Masked')
    axes[1, i].axis('off')
    
    # Reconstruction
    axes[2, i].imshow(reconstructed[i].squeeze(), cmap='gray')
    axes[2, i].set_title('Reconstructed')
    axes[2, i].axis('off')

plt.tight_layout()
plt.show()

### Key Insight: Masked Autoencoding

By learning to reconstruct masked images, the model learns:
- **Local patterns** (textures, edges)
- **Global structure** (object shapes)
- **Semantic understanding** (what's likely to be in masked regions)

These representations transfer extremely well to downstream tasks!

## 6. SimCLR: Contrastive Learning

### Intuition

**SimCLR (Simple Framework for Contrastive Learning)** learns representations by contrasting different views of the same image. The key idea:

- **Positive pairs**: Different augmentations of the same image should have similar representations
- **Negative pairs**: Different images should have dissimilar representations

This pushes the model to learn **augmentation-invariant features** — features that capture the essence of an image regardless of specific transformations.

### SimCLR Augmentations

Define strong augmentations that create different views of the same image.

In [ ]:
class SimCLRTransform:
    """Create two random augmented views of the same image."""
    
    def __init__(self):
        self.transform = transforms.Compose([
            transforms.RandomRotation(10),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
            transforms.GaussianBlur(kernel_size=3),
        ])
    
    def __call__(self, x):
        return self.transform(x), self.transform(x)

class SimCLRDataset(Dataset):
    """Dataset that returns two augmented views of each image."""
    
    def __init__(self, base_dataset):
        self.base_dataset = base_dataset
        self.transform = SimCLRTransform()
    
    def __len__(self):
        return len(self.base_dataset)
    
    def __getitem__(self, idx):
        image, _ = self.base_dataset[idx]
        view1, view2 = self.transform(image)
        return view1, view2

simclr_train = SimCLRDataset(train_dataset)
simclr_loader = DataLoader(simclr_train, batch_size=256, shuffle=True)

### Visualize Augmented Pairs

See how SimCLR creates different views of the same image.

In [ ]:
# Get augmented pairs
view1, view2 = next(iter(simclr_loader))

fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for i in range(4):
    # View 1
    axes[0, i].imshow(view1[i].squeeze(), cmap='gray')
    axes[0, i].set_title(f'View 1 (Image {i})')
    axes[0, i].axis('off')
    
    # View 2
    axes[1, i].imshow(view2[i].squeeze(), cmap='gray')
    axes[1, i].set_title(f'View 2 (Image {i})')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

print("These pairs should have similar representations despite different augmentations.")

### Build SimCLR Model

A CNN encoder with a projection head for contrastive learning.

In [ ]:
class SimCLR(nn.Module):
    """SimCLR model with encoder and projection head."""
    
    def __init__(self, embedding_dim=128):
        super().__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Flatten(),
        )
        
        # Projection head (important for contrastive learning)
        self.projection = nn.Sequential(
            nn.Linear(64 * 7 * 7, 256),
            nn.ReLU(),
            nn.Linear(256, embedding_dim)
        )
    
    def forward(self, x):
        features = self.encoder(x)
        projections = self.projection(features)
        # Normalize for cosine similarity
        return F.normalize(projections, dim=1)

simclr_model = SimCLR().to(device)
print(f"Parameters: {sum(p.numel() for p in simclr_model.parameters()):,}")

### NT-Xent Loss (Normalized Temperature-scaled Cross Entropy)

The contrastive loss used in SimCLR. It pulls positive pairs together and pushes negative pairs apart.

In [ ]:
def nt_xent_loss(z1, z2, temperature=0.5):
    """
    NT-Xent (Normalized Temperature-scaled Cross Entropy Loss).
    
    Args:
        z1, z2: Normalized embeddings of shape (batch_size, embedding_dim)
        temperature: Temperature parameter for scaling
    """
    batch_size = z1.shape[0]
    
    # Concatenate embeddings
    z = torch.cat([z1, z2], dim=0)  # Shape: (2*batch_size, embedding_dim)
    
    # Compute similarity matrix (cosine similarity)
    sim_matrix = torch.mm(z, z.t()) / temperature  # Shape: (2*batch_size, 2*batch_size)
    
    # Remove diagonal (self-similarity)
    mask = torch.eye(2 * batch_size, dtype=torch.bool, device=z.device)
    sim_matrix = sim_matrix.masked_fill(mask, -9e15)
    
    # Positive pairs: (i, i+batch_size) and (i+batch_size, i)
    pos_sim = torch.cat([sim_matrix[i, i+batch_size].unsqueeze(0) 
                        for i in range(batch_size)] +
                       [sim_matrix[i+batch_size, i].unsqueeze(0) 
                        for i in range(batch_size)])
    
    # Compute loss (cross-entropy)
    # For each sample, positive pair should have highest similarity
    loss = 0
    for i in range(2 * batch_size):
        # Get positive index
        pos_idx = (i + batch_size) % (2 * batch_size)
        
        # Numerator: exp(sim with positive)
        numerator = torch.exp(sim_matrix[i, pos_idx])
        
        # Denominator: sum of exp(sim with all others)
        denominator = torch.exp(sim_matrix[i]).sum() - torch.exp(sim_matrix[i, i])
        
        loss += -torch.log(numerator / denominator)
    
    return loss / (2 * batch_size)

### Train SimCLR

Train the model using contrastive learning.

In [ ]:
optimizer = optim.Adam(simclr_model.parameters(), lr=0.001)

epochs = 3

for epoch in range(epochs):
    simclr_model.train()
    epoch_loss = 0
    
    for view1, view2 in simclr_loader:
        view1, view2 = view1.to(device), view2.to(device)
        
        optimizer.zero_grad()
        
        # Get embeddings for both views
        z1 = simclr_model(view1)
        z2 = simclr_model(view2)
        
        # Compute contrastive loss
        loss = nt_xent_loss(z1, z2)
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(simclr_loader)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")

### Key Insight: SimCLR

SimCLR learns representations by making augmented views of the same image similar while keeping different images dissimilar. This produces **augmentation-invariant features** that capture semantic content rather than superficial details.

Key components:
1. **Strong augmentations** — create diverse views
2. **Projection head** — helps with contrastive learning
3. **Large batch sizes** — more negative examples improve learning
4. **Temperature scaling** — controls the concentration of the distribution

## 7. BERT-Style Masking

### Intuition

**BERT (Bidirectional Encoder Representations from Transformers)** pioneered masked language modeling for text. The idea:

- Randomly mask some words in a sentence
- Train the model to predict the masked words
- Use bidirectional context (words before AND after)

This forces the model to learn deep language understanding — grammar, semantics, and context.

### Create Text Data

We'll use simple sentences for demonstration.

In [ ]:
# Sample sentences
sentences = [
    "the cat sat on the mat",
    "a dog ran in the park",
    "the bird flew over the tree",
    "a fish swam in the water",
    "the mouse ate some cheese",
] * 200  # Repeat for more training data

# Build vocabulary
all_words = set(' '.join(sentences).split())
vocab = ['<PAD>', '<MASK>'] + sorted(list(all_words))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for i, w in enumerate(vocab)}

print(f"Vocabulary size: {len(vocab)}")
print(f"Words: {vocab}")

### Create BERT Masking Dataset

Randomly mask 15% of words and train model to predict them.

In [ ]:
class BERTMaskingDataset(Dataset):
    """Dataset that applies BERT-style masking to sentences."""
    
    def __init__(self, sentences, word2idx, mask_prob=0.15):
        self.sentences = sentences
        self.word2idx = word2idx
        self.mask_prob = mask_prob
        self.mask_token = word2idx['<MASK>']
    
    def __len__(self):
        return len(self.sentences)
    
    def __getitem__(self, idx):
        words = self.sentences[idx].split()
        tokens = [self.word2idx[w] for w in words]
        
        # Create masked version
        masked_tokens = tokens.copy()
        labels = [-100] * len(tokens)  # -100 is ignored in loss
        
        # Randomly mask tokens
        for i in range(len(tokens)):
            if random.random() < self.mask_prob:
                labels[i] = tokens[i]  # Save original token
                masked_tokens[i] = self.mask_token  # Mask it
        
        return torch.tensor(masked_tokens), torch.tensor(labels)

bert_train = BERTMaskingDataset(sentences, word2idx)
bert_loader = DataLoader(bert_train, batch_size=32, shuffle=True)

### Visualize Masked Sentences

See what BERT-style masking looks like.

In [ ]:
# Get a sample
masked_tokens, labels = bert_train[0]

print("Original sentence:", sentences[0])
print("\nMasked sentence:", ' '.join([idx2word[t.item()] for t in masked_tokens]))
print("\nMasked positions and targets:")
for i, label in enumerate(labels):
    if label != -100:
        print(f"  Position {i}: [MASK] -> {idx2word[label.item()]}")

### Build Simple BERT Model

A simple embedding + transformer layer for masked language modeling.

In [ ]:
class SimpleBERT(nn.Module):
    """Simple BERT-style model for masked language modeling."""
    
    def __init__(self, vocab_size, embedding_dim=64, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # Simple feedforward "transformer" (not using true self-attention for simplicity)
        self.encoder = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
        )
        
        # Output layer predicts vocabulary
        self.output = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, tokens):
        # Embed tokens
        x = self.embedding(tokens)  # (batch, seq_len, embedding_dim)
        
        # Encode
        x = self.encoder(x)  # (batch, seq_len, hidden_dim)
        
        # Predict
        logits = self.output(x)  # (batch, seq_len, vocab_size)
        
        return logits

bert_model = SimpleBERT(len(vocab)).to(device)
print(f"Parameters: {sum(p.numel() for p in bert_model.parameters()):,}")

### Train BERT Model

Train to predict masked words.

In [ ]:
optimizer = optim.Adam(bert_model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=-100)  # Ignore non-masked positions

epochs = 5

for epoch in range(epochs):
    bert_model.train()
    epoch_loss = 0
    
    for masked_tokens, labels in bert_loader:
        masked_tokens = masked_tokens.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        logits = bert_model(masked_tokens)  # (batch, seq_len, vocab_size)
        
        # Reshape for loss computation
        loss = criterion(logits.view(-1, len(vocab)), labels.view(-1))
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(bert_loader)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")

### Test Masked Prediction

See if the model can predict masked words correctly.

In [ ]:
bert_model.eval()
with torch.no_grad():
    # Get a sample
    masked_tokens, labels = bert_train[0]
    masked_tokens = masked_tokens.unsqueeze(0).to(device)  # Add batch dimension
    
    # Predict
    logits = bert_model(masked_tokens)
    predictions = logits.argmax(dim=-1).squeeze().cpu()
    
    print("Original:", sentences[0])
    print("Masked:", ' '.join([idx2word[t.item()] for t in bert_train[0][0]]))
    print("Predicted:", ' '.join([idx2word[t.item()] for t in predictions]))
    
    # Check masked positions
    print("\nMasked word predictions:")
    for i, label in enumerate(labels):
        if label != -100:
            true_word = idx2word[label.item()]
            pred_word = idx2word[predictions[i].item()]
            match = "✓" if true_word == pred_word else "✗"
            print(f"  Position {i}: {pred_word} (true: {true_word}) {match}")

### Key Insight: BERT-Style Masking

Masked language modeling forces the model to learn:
- **Contextual representations** — word meaning depends on context
- **Bidirectional understanding** — use both left and right context
- **Semantic relationships** — which words fit in which contexts

This is why BERT revolutionized NLP — the learned representations work well for almost any language task!

## 8. Key Takeaways

### Self-Supervised Learning Principles

All self-supervised learning methods share these principles:

1. **Automatic Label Generation** — Labels come from data structure, not human annotation
2. **Pretext Tasks** — Artificial tasks that force learning useful representations
3. **Transfer Learning** — Learned representations transfer to downstream tasks
4. **Scalability** — Can leverage massive unlabeled datasets

### Method Comparison

| Method | Pretext Task | What It Learns | Best For |
|--------|--------------|----------------|----------|
| **Rotation Prediction** | Predict rotation angle | Spatial features, orientation | Vision |
| **Jigsaw Puzzles** | Predict patch ordering | Object parts, spatial context | Vision |
| **Masked Autoencoding** | Reconstruct masked regions | Local & global structure | Vision |
| **SimCLR** | Match augmented views | Augmentation-invariant features | Vision |
| **BERT Masking** | Predict masked words | Contextual word representations | NLP |

### When to Use Self-Supervised Learning

Self-supervised learning is ideal when:
- You have lots of unlabeled data but little labeled data
- Labels are expensive or impossible to obtain
- You want to pretrain models for transfer learning
- You need robust, generalizable representations

### Modern Impact

Self-supervised learning powers today's most successful models:
- **GPT** (language modeling)
- **BERT** (masked language modeling)
- **MAE** (masked autoencoding for vision)
- **SimCLR, MoCo** (contrastive learning)

The trend is clear: **self-supervised pretraining + supervised fine-tuning** is the path to state-of-the-art performance across domains.

## Experiments & Exploration

### Ideas to Try

1. **Compare pretext tasks**: Train different self-supervised methods on the same dataset, then evaluate their learned representations on a classification task. Which method learns the best features?

2. **Augmentation sensitivity**: For SimCLR, try different augmentation strengths. What happens with very weak augmentations? Very strong ones?

3. **Mask ratio ablation**: In masked autoencoding, try different masking ratios (25%, 50%, 75%). How does it affect reconstruction quality and learned representations?

4. **Transfer learning**: Use the pretrained encoder from any method as a feature extractor for MNIST classification. Does self-supervised pretraining help?

5. **Combine methods**: Can you use multiple pretext tasks together (e.g., rotation + jigsaw)? Does this improve representations?

The beauty of self-supervised learning is that **data is the only limit** — more data means better representations, no labeling required!